In [1]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub

# Mediapipe face and pose setup
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose
face_mesh = mp_face_mesh.FaceMesh(min_detection_confidence=0.7, min_tracking_confidence=0.7)
pose = mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7)

# Load SSD MobileNet model from TensorFlow Hub
ssd_model = hub.load("https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2")

# Define classes for COCO dataset (only relevant classes for this task)
coco_classes = {
    1: "person", 44: "bottle", 45: "wine glass", 46: "cup", 47: "fork",
    48: "knife", 49: "spoon", 77: "cell phone", 85: "toothbrush", 88: "gun"
}
relevant_classes = ["knife", "gun"]

# Utility function to preprocess frame for SSD
def preprocess_frame(frame):
    resized_frame = tf.image.resize(frame, (300, 300))
    resized_frame = tf.cast(resized_frame, tf.uint8)
    return tf.expand_dims(resized_frame, axis=0)

# Utility function to draw bounding boxes
def draw_boxes(frame, detections, scores, classes, threshold=0.25):
    """Draw bounding boxes on the frame."""
    h, w, _ = frame.shape
    for i in range(len(scores)):
        if scores[i] > threshold:
            class_id = int(classes[i])
            if class_id in coco_classes and coco_classes[class_id] in relevant_classes:
                ymin, xmin, ymax, xmax = detections[i]
                (x1, y1, x2, y2) = int(xmin * w), int(ymin * h), int(xmax * w), int(ymax * h)

                # Draw the bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

                # Add label and confidence
                label = coco_classes[class_id]
                confidence = scores[i]
                cv2.putText(frame, f"{label} {confidence:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
           
def calculate_face_coverage(face_landmarks, brightness):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14] 
    total_landmarks = len(critical_landmarks)
    visible_landmarks = 0

    visibility_threshold = 0.6 - (brightness / 255.0) * 0.2
    z_threshold_min = -0.1 - (brightness / 255.0) * 0.05
    z_threshold_max = 0.1 + (brightness / 255.0) * 0.05

    for idx in critical_landmarks:
        landmark = face_landmarks[idx]
        if z_threshold_min < landmark.z < z_threshold_max:
  
            visible_landmarks += 1

    coverage = (visible_landmarks / total_landmarks) * 100
    return coverage

def analyze_behavior(pose_landmarks):
    left_hand_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_hand_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    if left_hand_y < nose_y or right_hand_y < nose_y:
        return True  
    return False


cap = cv2.VideoCapture(0)
alert_message = "Warning: Potential Threat!"
alert_duration = 3
last_alert_time = 0

while True:
    success, frame = cap.read()
    if not success:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray_frame)

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(image_rgb)
    pose_results = pose.process(image_rgb)

    unusual_behavior_detected = False

    if face_results.multi_face_landmarks:
        for face_landmarks in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(face_landmarks.landmark, brightness)
            print(f"Coverage: {coverage:.2f}% (Brightness: {brightness:.2f})")

            if coverage < 20:
                current_time = time.time()
                if current_time - last_alert_time > alert_duration:
                    cv2.putText(frame, alert_message, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

                    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
                    filename = f"warning_image_{timestamp}.jpg"
                    cv2.imwrite(filename, frame)
                    last_alert_time = current_time

    if pose_results.pose_landmarks:
        unusual_behavior_detected = analyze_behavior(pose_results.pose_landmarks.landmark)

    if unusual_behavior_detected:
        cv2.putText(frame, "Unusual Behavior Detected!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        filename = f"behavior_warning_{timestamp}.jpg"
        cv2.imwrite(filename, frame)

    # Object detection using SSD
    preprocessed_frame = preprocess_frame(frame)
    result = ssd_model(preprocessed_frame)
    detections = result["detection_boxes"].numpy()[0]  # Bounding boxes
    scores = result["detection_scores"].numpy()[0]  # Confidence scores
    classes = result["detection_classes"].numpy()[0]  # Class IDs

    # Draw boxes for relevant classes
    draw_boxes(frame, detections, scores, classes, threshold=0.5)

    cv2.imshow('Surveillance System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Coverage: 100.00% (Brightness: 204.60)
Coverage: 100.00% (Brightness: 142.27)
Coverage: 100.00% (Brightness: 142.31)
Coverage: 100.00% (Brightness: 142.37)
Coverage: 100.00% (Brightness: 142.24)
Coverage: 100.00% (Brightness: 142.29)
Coverage: 100.00% (Brightness: 142.31)
Coverage: 100.00% (Brightness: 142.36)
Coverage: 100.00% (Brightness: 142.46)
Coverage: 100.00% (Brightness: 142.47)
Coverage: 100.00% (Brightness: 142.42)
Coverage: 100.00% (Brightness: 142.55)
Coverage: 100.00% (Brightness: 142.48)
Coverage: 100.00% (Brightness: 142.47)
Coverage: 100.00% (Brightness: 142.47)
Coverage: 100.00% (Brightness: 142.49)
Coverage: 100.00% (Brightness: 142.44)
Coverage: 100.00% (Brightness: 142.26)
Coverage: 100.00% (Brightness: 143.02)
Coverage: 100.00% (Brightness: 143.11)
Coverage: 100.00% (Brightness: 142.84)
Coverage: 100.00% (Brightness: 142.75)
Coverage: 100.00% (Brightness: 142.78)
Coverage: 100.00% (Brightness: 142.70)
Coverage: 100.00% (Brightness: 142.63)
Coverage: 100.00% (Bright